# Release Volume Sampler: Operational steps.

This notebook is used to run the operational part of the workflow interactively. That is, to assign probabilities to release volumes based on shakemaps.

In [ ]:
# Imports
import os
import numpy as np
import rasterio

# Change to the src directory
# This is necessary to ensure that the script runs in the correct context
# and can find the necessary modules and files.
os.chdir("/home/ebr/projects/release-volume-sampler/src")

In [ ]:
from rvsampler.displacements import DisplacementProbabilityAggregator
from rvsampler.shakemaps_reader import ShakemapsReader
from rvsampler.utils import create_dir
from rvsampler.set_logg import setup_logger
from rvsampler.database_handler import VolumeDatabaseHandler
from rvsampler.aggregate import ProbabilityAggregator

## Computation of release probabilities (by shakemap).

See below routine for aggregation over all shakemaps.

In [ ]:
# Set region and directories
region = "messina_002"  # Change as needed
rootdir = "/home/ebr/projects/release-volume-sampler"  # Change as needed
rundir = os.path.join(rootdir, 'generated', region)
logger = setup_logger("operational", rundir)

logger.info(f"Running operational notebook for region {region} in {rundir}")

In [ ]:
# Preprocess shakemaps
shakemaps_filename = os.path.join(rootdir, "input/shakemaps/PGA_data/H_Z_pda_data_log10_G.json")
source_parameters_filename = os.path.join(rootdir, "input/shakemaps/messina_1908/source_parameters.csv")
bathymetry_filename = os.path.join(rundir, "bathy_truncated.tif")

shakemaps_reader = ShakemapsReader(
    shakemaps_filename=shakemaps_filename,
    source_parameters_filename=source_parameters_filename,
    rundir=rundir,
)

with rasterio.open(bathymetry_filename) as src:
    bounds = src.bounds
    profile = src.profile.copy()

shakemaps_reader.write_shakemaps_to_rasters(
    profile=profile, 
    bounds=bounds,
    samples=None, # None (reads all samples) or specify a list of samples to read, e.g., [0,3]
    interpolation_method='linear', # “linear”, “nearest”, “slinear”, “cubic”, “quintic” and “pchip”
)
shakemaps_reader.completed()

In [ ]:
# Calculate Displacement probabilities
displacements_exceedance_params = {
    "cumulative_dir": os.path.join(rundir, "displacements"),
    "outfile_name": "exceedance_displacement.npz"
}
dpa = DisplacementProbabilityAggregator(rundir, magnitude=7)
dpa.compute_probabilities_by_sample(displacement_threshold=5., nr_of_pga_thresholds=100)
dpa.completed()

In [ ]:
# Write probabilities to database
displacements_dir = os.path.join(rundir, "displacements")

with VolumeDatabaseHandler(rundir) as volumes_db:
    for fname in os.listdir(displacements_dir):
        if fname.startswith("sample_") and os.path.isdir(os.path.join(displacements_dir, fname)):
            sample_nr = fname.split("_")[-1]
            column_name = f"p_shake_{sample_nr}"
            volumes_db.assign_probabilities_to_seed_triangles(
                displacement_dir=os.path.join(displacements_dir, fname),
                table_filename="exceedance_displacement.npz",
                column_name=column_name
            )


In [ ]:
# Compute cluster release probabilities
pag = ProbabilityAggregator(rundir)
result = pag.compute_cluster_release_probabilities()
pag.plot_cluster_probability_heatmap(save_fig=True);
pag.completed()

##  Computation of release probability based on cumulative shakemap.

In [ ]:
# Set region and directories
region = "messina_001"  # Change as needed
rootdir = "/home/ebr/projects/release-volume-sampler"  # Change as needed
rundir = os.path.join(rootdir, 'generated', region)
logger = setup_logger("operational", rundir)

logger.info(f"Running operational notebook for region {region} in {rundir}")

In [ ]:
# Preprocess shakemaps
shakemaps_filename = os.path.join(rootdir, "input/shakemaps/PGA_data/H_Z_pda_data_log10_G.json")
source_parameters_filename = os.path.join(rootdir, "input/shakemaps/messina_1908/source_parameters.csv")
bathymetry_filename = os.path.join(rundir, "bathy_truncated.tif")

shakemaps_reader = ShakemapsReader(
    shakemaps_filename=shakemaps_filename,
    source_parameters_filename=source_parameters_filename,
    rundir=rundir,
)

with rasterio.open(bathymetry_filename) as src:
    bounds = src.bounds
    profile = src.profile.copy()


shakemaps_reader.write_cumulative_distribution(
    profile=profile,
    bounds=bounds,
    thresholds=np.linspace(-3, 0, 10),  # Array of thresholds for cumulative distribution
    interpolation_method='linear'
)

In [ ]:
# Calculate Displacement probabilities
displacements_exceedance_params = {
    "cumulative_dir": os.path.join(rundir, "displacements"),
    "outfile_name": "exceedance_displacement.npz"
}

#thresholds = np.arange(1, 10, step=2.) # Displacement thresholds in cm.
dpa = DisplacementProbabilityAggregator(rundir, magnitude=7)
dpa.compute_aggregated_probabilities(displacement_threshold=5.)

In [ ]:
with VolumeDatabaseHandler(rundir) as volumes_db:
    volumes_db.assign_probabilities_to_seed_triangles(
        displacement_dir=os.path.join(rundir,"displacements","cumulative"),
        table_filename="exceedance_displacement.npz",
        column_name="p_shake_cum"
    )